# Preprocesamiento y representación de texto

## Objetivo
Hasta ahora trabajaste con spaCy para tokenizar, lematizar, etiquetar y reconocer entidades. Todas esas operaciones producen *información sobre el texto*, pero todavía no decidimos **cómo convertir un texto en una representación estructurada** que sirva como entrada para análisis posteriores.

En este cuaderno vas a:
- recorrer paso a paso las decisiones de preprocesamiento que transforman texto crudo en una lista de tokens;
- comparar distintos pipelines de preprocesamiento y observar cómo cada uno altera los resultados;
- ver en la práctica cómo otra herramienta (Stanza) produce resultados distintos sobre el mismo texto;
- entender que la representación final no es neutral: cada decisión (lematizar o no, filtrar stopwords o no, qué herramienta usar) descarta información y puede introducir sesgos.

## Resultados de aprendizaje

Al finalizar este cuaderno vas a poder:
- construir distintos pipelines de preprocesamiento en spaCy y explicar las diferencias entre ellos;
- identificar cómo los errores de lematización (por ejemplo, frente al voseo) se propagan al análisis de frecuencias;
- usar Stanza como alternativa a spaCy e integrarla vía `spacy-stanza` sin reescribir el pipeline;
- justificar qué pipeline elegirías para una tarea concreta y por qué.

## Qué NO cubre este cuaderno
Todavía no vamos a desarrollar TF-IDF ni ningún esquema de ponderación numérica. Eso se trabaja en el próximo cuaderno. Pero para que la mención no quede en el aire, una intuición rápida:

**TF-IDF** (*Term Frequency – Inverse Document Frequency*) es un método para asignarle un **peso numérico** a cada palabra en un documento. La idea básica es: una palabra es importante para un documento si aparece mucho en *ese* documento (TF alta) pero poco en el *resto* de los documentos (IDF alta). Así, palabras como "de" o "la" — que aparecen en todos lados — reciben peso bajo, mientras que términos específicos del tema reciben peso alto.

Para que TF-IDF funcione, necesita recibir una **lista de tokens** por cada documento. Esa lista es exactamente lo que vamos a aprender a construir acá. El foco de este cuaderno está en **qué le damos de comer** a esos métodos.

## Relación con los cuadernos anteriores
- En `001` a `003` construiste la base con el pipeline estándar de spaCy.
- En `004` diagnosticaste fallas del modelo frente al español rioplatense.
- En `005` integraste todo en un analizador de noticias.
- Acá vas a formalizar el paso que conecta el texto crudo con cualquier análisis cuantitativo posterior.

---
## 1. Carga de spaCy y texto de ejemplo

Arrancamos cargando el modelo de español de spaCy y preparando un texto de ejemplo en español rioplatense. Fijate que también agregamos manualmente la palabra "él" a la lista de stopwords, ya que esta versión del modelo no la incluye por defecto.

In [5]:
import pandas as pd
from IPython.display import display

In [6]:
import spacy
from collections import Counter

nlp = spacy.load("es_core_news_sm")

# Aseguramos que "él" sea tratado como stopword
from spacy.lang.es.stop_words import STOP_WORDS
STOP_WORDS.add("él")
print(f"Modelo cargado: {nlp.meta['name']}")
print(f"Pipeline activo: {nlp.pipe_names}")

Modelo cargado: core_news_sm
Pipeline activo: ['tok2vec', 'morphologizer', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


Procesamos el texto de ejemplo con el pipeline. `nlp(texto)` recorre todo el pipeline: tokeniza, asigna etiquetas morfológicas, analiza la sintaxis y detecta entidades. El resultado es un objeto `Doc` que contiene toda esa información por token.

In [7]:
# Texto de ejemplo en español rioplatense
texto = (
    "Vos sabés que si querés venís mañana a la clase de NLP. "
    "La clase es sobre procesamiento de texto y representación de datos. "
    "Tenés que traer la notebook con spaCy instalado. "
    "Si no podés mañana, avisá y coordinamos otro día. "
    "El procesamiento de lenguaje natural es clave para entender datos textuales."
)

doc = nlp(texto)
print(f"Texto procesado: {len(doc)} tokens")
print(f"Oraciones detectadas: {len(list(doc.sents))}")

Texto procesado: 57 tokens
Oraciones detectadas: 5


El modelo detectó 57 tokens y 5 oraciones, que coinciden con los puntos del texto. Ese recuento incluye puntuación y stopwords. En los pasos siguientes vamos a ver qué pasa cuando aplicamos distintos criterios de filtrado.

---
## 2. Extracción de representaciones básicas

Antes de decidir qué pipeline usar, necesitamos ver qué información nos da spaCy para cada token. Vamos a extraer cuatro vistas distintas del mismo texto:

1. **Tokens originales** — el texto tal como aparece.
2. **Tokens en minúscula** — normalización básica.
3. **Lemas** — forma canónica según el modelo.
4. **Tokens filtrados** — sin stopwords ni puntuación.

### 2.1 Tokens originales

`token.text` devuelve la forma exacta del token, tal como aparece en el texto fuente. Es el punto de partida antes de cualquier transformación.

In [8]:
# 2.1 Tokens originales
tokens_originales = [token.text for token in doc]
print("TOKENS ORIGINALES:")
print(tokens_originales)
print(f"Total: {len(tokens_originales)}")

TOKENS ORIGINALES:
['Vos', 'sabés', 'que', 'si', 'querés', 'venís', 'mañana', 'a', 'la', 'clase', 'de', 'NLP', '.', 'La', 'clase', 'es', 'sobre', 'procesamiento', 'de', 'texto', 'y', 'representación', 'de', 'datos', '.', 'Tenés', 'que', 'traer', 'la', 'notebook', 'con', 'spaCy', 'instalado', '.', 'Si', 'no', 'podés', 'mañana', ',', 'avisá', 'y', 'coordinamos', 'otro', 'día', '.', 'El', 'procesamiento', 'de', 'lenguaje', 'natural', 'es', 'clave', 'para', 'entender', 'datos', 'textuales', '.']
Total: 57


### 2.2 Tokens en minúscula

Pasamos todo a minúscula con `token.text.lower()`. Esto unifica `"La"` y `"la"`, o `"NLP"` y `"nlp"`, pero pierde información sobre mayúsculas que puede ser relevante (nombres propios, siglas).

In [9]:
# 2.2 Tokens en minúscula
tokens_lower = [token.text.lower() for token in doc]
print("TOKENS LOWERCASED:")
print(tokens_lower)
print(f"Total: {len(tokens_lower)}")

TOKENS LOWERCASED:
['vos', 'sabés', 'que', 'si', 'querés', 'venís', 'mañana', 'a', 'la', 'clase', 'de', 'nlp', '.', 'la', 'clase', 'es', 'sobre', 'procesamiento', 'de', 'texto', 'y', 'representación', 'de', 'datos', '.', 'tenés', 'que', 'traer', 'la', 'notebook', 'con', 'spacy', 'instalado', '.', 'si', 'no', 'podés', 'mañana', ',', 'avisá', 'y', 'coordinamos', 'otro', 'día', '.', 'el', 'procesamiento', 'de', 'lenguaje', 'natural', 'es', 'clave', 'para', 'entender', 'datos', 'textuales', '.']
Total: 57


### 2.3 Lemas

`token.lemma_` devuelve la forma canónica o *lema* del token según el modelo. Por ejemplo, "datos" → "dato", "coordinamos" → "coordinar". Prestá atención a las formas del voseo: el modelo no siempre las lematiza correctamente.

In [10]:
# 2.3 Lemas
lemas = [token.lemma_ for token in doc]
print("LEMAS:")
print(lemas)
print(f"Total: {len(lemas)}")

LEMAS:
['vos', 'sabé', 'que', 'si', 'querés', 'venís', 'mañana', 'a', 'el', 'clase', 'de', 'NLP', '.', 'el', 'clase', 'ser', 'sobre', 'procesamiento', 'de', 'texto', 'y', 'representación', 'de', 'dato', '.', 'tenés', 'que', 'traer', 'el', 'notebook', 'con', 'spacy', 'instalado', '.', 'si', 'no', 'podés', 'mañana', ',', 'avisá', 'y', 'coordinar', 'otro', 'día', '.', 'el', 'procesamiento', 'de', 'lenguaje', 'natural', 'ser', 'clave', 'para', 'entender', 'dato', 'textual', '.']
Total: 57


In [ ]:
# Organizamos los lemas en filas de 10 para mejor visualización
N = 10
filas = [lemas[i:i + N] for i in range(0, len(lemas), N)]

tabla_lemas = pd.DataFrame(filas)
tabla_lemas.index = [
    f"{i * N}–{min((i + 1) * N - 1, len(lemas) - 1)}"
    for i in range(len(filas))
]
tabla_lemas.columns = [str(j + 1) for j in range(N)]
tabla_lemas.index.name = "pos"

display(tabla_lemas.fillna(""))


Observá que formas como `"querés"`, `"venís"` o `"tenés"` no fueron lematizadas correctamente: el modelo las devuelve casi idénticas a la forma original, porque el corpus de entrenamiento (AnCora, con predominio del español peninsular) tiene poca representación del voseo. Eso es exactamente lo que analizamos en el cuaderno `004`.

### 2.4 Tokens filtrados

Aplicamos dos filtros simultáneos:
- `token.is_alpha`: conserva solo tokens que contienen exclusivamente letras (descarta signos de puntuación, números, etc.).
- `not token.is_stop`: descarta palabras que el modelo marcó como stopwords (artículos, preposiciones, conjunciones, etc.).

In [11]:
# 2.4 Tokens filtrados (solo alfabéticos, sin stopwords)
tokens_filtrados = [
    token.text for token in doc
    if token.is_alpha and not token.is_stop
]
print("TOKENS FILTRADOS (alfabéticos, sin stopwords):")
print(tokens_filtrados)
print(f"Total: {len(tokens_filtrados)} (de {len(tokens_originales)} originales)")

TOKENS FILTRADOS (alfabéticos, sin stopwords):
['Vos', 'sabés', 'querés', 'venís', 'mañana', 'clase', 'NLP', 'clase', 'procesamiento', 'texto', 'representación', 'datos', 'Tenés', 'traer', 'notebook', 'spaCy', 'instalado', 'podés', 'mañana', 'avisá', 'coordinamos', 'procesamiento', 'lenguaje', 'natural', 'clave', 'entender', 'datos', 'textuales']
Total: 28 (de 57 originales)


In [12]:
# Organizamos los tokens filtrados en filas de 7 para mejor visualización
# (28 tokens → 4 filas prolijas)
N = 7
filas = [tokens_filtrados[i:i + N] for i in range(0, len(tokens_filtrados), N)]

tabla_filtrados = pd.DataFrame(filas)
tabla_filtrados.index = [
    f"{i * N}–{min((i + 1) * N - 1, len(tokens_filtrados) - 1)}"
    for i in range(len(filas))
]
tabla_filtrados.columns = [str(j + 1) for j in range(N)]
tabla_filtrados.index.name = "pos"

display(tabla_filtrados.fillna(""))


,1,2,3,4,5,6,7
pos,,,,,,,
0–6,Vos,sabés,querés,venís,mañana,clase,NLP
7–13,clase,procesamiento,texto,representación,datos,Tenés,traer
14–20,notebook,spaCy,instalado,podés,mañana,avisá,coordinamos
21–27,procesamiento,lenguaje,natural,clave,entender,datos,textuales


Observá las diferencias entre las cuatro representaciones:
- Pasar a *lowercase* unifica mayúsculas y minúsculas, pero pierde información sobre nombres propios o siglas.
- Los lemas reducen variantes flexivas a una forma base, pero dependen de la calidad del modelo (recordá lo que vimos en el cuaderno `004` sobre el voseo).
- El filtrado elimina ruido, pero también descarta palabras funcionales que pueden tener valor en ciertos análisis.

Pasamos de 57 tokens a 28 después del filtrado. ¿Qué descartamos? Principalmente puntuación, artículos, preposiciones y conjunciones. ¿Qué conservamos? Las palabras que, en principio, aportan mayor contenido semántico.

---
## 3. Comparación de pipelines de preprocesamiento

Ahora vamos a definir tres estrategias distintas de preprocesamiento y comparar sus resultados sobre el mismo texto. Cada pipeline es una función que recibe un `Doc` de spaCy y devuelve una lista de strings.

La pregunta que guía esta sección es: **¿importa qué pipeline elegimos?** Como vas a ver, la respuesta es sí, y de manera que puede ser difícil de detectar si no la examinamos explícitamente.

### Pipeline A: naive (lowercase + solo alfabéticos)

La estrategia más simple: pasar todo a minúscula y conservar solo tokens que contengan letras. No usa información lingüística del modelo: no lematiza ni filtra stopwords.

In [13]:
def pipeline_a(doc):
    """Pipeline naive: lowercase + solo palabras alfabéticas."""
    return [
        token.text.lower()
        for token in doc
        if token.is_alpha
    ]

resultado_a = pipeline_a(doc)
print("PIPELINE A (naive):")
print(resultado_a)
print(f"\nTotal tokens: {len(resultado_a)}")

freq_a = Counter(resultado_a)
print("\nTop 10 palabras más frecuentes:")
for palabra, cuenta in freq_a.most_common(10):
    print(f"  {palabra:20} -> {cuenta}")

PIPELINE A (naive):
['vos', 'sabés', 'que', 'si', 'querés', 'venís', 'mañana', 'a', 'la', 'clase', 'de', 'nlp', 'la', 'clase', 'es', 'sobre', 'procesamiento', 'de', 'texto', 'y', 'representación', 'de', 'datos', 'tenés', 'que', 'traer', 'la', 'notebook', 'con', 'spacy', 'instalado', 'si', 'no', 'podés', 'mañana', 'avisá', 'y', 'coordinamos', 'otro', 'día', 'el', 'procesamiento', 'de', 'lenguaje', 'natural', 'es', 'clave', 'para', 'entender', 'datos', 'textuales']

Total tokens: 51

Top 10 palabras más frecuentes:
  de                   -> 4
  la                   -> 3
  que                  -> 2
  si                   -> 2
  mañana               -> 2
  clase                -> 2
  es                   -> 2
  procesamiento        -> 2
  y                    -> 2
  datos                -> 2


El top 10 está dominado por palabras funcionales ("de", "la", "que", "es") que no aportan información sobre el contenido del texto. Este es el problema clásico del pipeline naive: sin filtrar stopwords, las palabras más frecuentes dicen poco sobre el tema.

### Pipeline B: lingüístico (lemas + sin stopwords)

Usa la información del modelo para lematizar y eliminar stopwords. Es la estrategia más habitual en tutoriales de NLP. Fijate que ahora el resultado cambia: se van las palabras funcionales y quedan las que, en teoría, tienen más carga semántica.

In [16]:
def pipeline_b(doc):
    """Pipeline lingüístico: lemas, sin stopwords, solo alfabéticos."""
    return [
        token.lemma_.lower()
        for token in doc
        if token.is_alpha and not token.is_stop
    ]

resultado_b = pipeline_b(doc)
print("PIPELINE B (lingüístico):")
print(resultado_b)
print(f"\nTotal tokens: {len(resultado_b)}")

freq_b = Counter(resultado_b)
print("\nTop 10 palabras más frecuentes:")
for palabra, cuenta in freq_b.most_common(10):
    print(f"  {palabra:20} -> {cuenta}")

PIPELINE B (lingüístico):
['vos', 'sabé', 'querés', 'venís', 'mañana', 'clase', 'nlp', 'clase', 'procesamiento', 'texto', 'representación', 'dato', 'tenés', 'traer', 'notebook', 'spacy', 'instalado', 'podés', 'mañana', 'avisá', 'coordinar', 'procesamiento', 'lenguaje', 'natural', 'clave', 'entender', 'dato', 'textual']

Total tokens: 28

Top 10 palabras más frecuentes:
  mañana               -> 2
  clase                -> 2
  procesamiento        -> 2
  dato                 -> 2
  vos                  -> 1
  sabé                 -> 1
  querés               -> 1
  venís                -> 1
  nlp                  -> 1
  texto                -> 1


Ahora el top muestra palabras con contenido real: "procesamiento", "clase", "mañana", "dato". Pero fijate que aparecen formas como `"sabé"`, `"querés"`, `"podés"` — lemas incorrectos para el voseo. El modelo no reconoció estas formas verbales y las devolvió sin lematizar correctamente.

### Pipeline C: ajustado (lemas corregidos para el voseo)

Igual al B, pero con un diccionario de corrección manual para formas del voseo que spaCy no lematiza bien. Esto retoma lo que diagnosticamos en el cuaderno `004`.

La lógica es simple: si el token en minúscula aparece en el diccionario de correcciones, usamos la corrección manual; si no, usamos el lema del modelo.

In [19]:
# Diccionario de correcciones para el voseo
CORRECCIONES_VOSEO = {
    "sos": "ser",
    "tenés": "tener",
    "sabés": "saber",
    "querés": "querer",
    "venís": "venir",
    "podés": "poder",
    "avisá": "avisar",
    "revisá": "revisar",
    "analizá": "analizar",
}


def pipeline_c(doc, correcciones=None):
    """Pipeline ajustado: lemas corregidos, sin stopwords, solo alfabéticos."""
    if correcciones is None:
        correcciones = {}

    resultado = []
    for token in doc:
        if not token.is_alpha or token.is_stop:
            continue

        forma = token.text.lower()
        # Si existe corrección manual, usarla; si no, usar el lema del modelo
        if forma in correcciones:
            lema = correcciones[forma]
        else:
            lema = token.lemma_.lower()

        resultado.append(lema)

    return resultado


resultado_c = pipeline_c(doc, CORRECCIONES_VOSEO)
print("PIPELINE C (ajustado):")
print(resultado_c)
print(f"\nTotal tokens: {len(resultado_c)}")

freq_c = Counter(resultado_c)
print("\nTop 10 palabras más frecuentes:")
for palabra, cuenta in freq_c.most_common(10):
    print(f"  {palabra:20} -> {cuenta}")

PIPELINE C (ajustado):
['vos', 'saber', 'querer', 'venir', 'mañana', 'clase', 'nlp', 'clase', 'procesamiento', 'texto', 'representación', 'dato', 'tener', 'traer', 'notebook', 'spacy', 'instalado', 'poder', 'mañana', 'avisar', 'coordinar', 'procesamiento', 'lenguaje', 'natural', 'clave', 'entender', 'dato', 'textual']

Total tokens: 28

Top 10 palabras más frecuentes:
  mañana               -> 2
  clase                -> 2
  procesamiento        -> 2
  dato                 -> 2
  vos                  -> 1
  saber                -> 1
  querer               -> 1
  venir                -> 1
  nlp                  -> 1
  texto                -> 1


Ahora los verbos del voseo aparecen con su lema correcto: `"saber"`, `"querer"`, `"venir"`, `"poder"`. La lista de frecuencias se mantiene igual en cuanto a los sustantivos más frecuentes, pero los verbos ahora están normalizados.

### Comparación lado a lado

In [20]:
print("=" * 60)
print("COMPARACIÓN DE PIPELINES")
print("=" * 60)

pipelines = {
    "A (naive)": resultado_a,
    "B (lingüístico)": resultado_b,
    "C (ajustado)": resultado_c,
}

for nombre, tokens in pipelines.items():
    freq = Counter(tokens)
    print(f"\n--- {nombre} ---")
    print(f"  Tokens totales:  {len(tokens)}")
    print(f"  Tokens únicos:   {len(set(tokens))}")
    print(f"  Top 5: {freq.most_common(5)}")

COMPARACIÓN DE PIPELINES

--- A (naive) ---
  Tokens totales:  51
  Tokens únicos:   38
  Top 5: [('de', 4), ('la', 3), ('que', 2), ('si', 2), ('mañana', 2)]

--- B (lingüístico) ---
  Tokens totales:  28
  Tokens únicos:   24
  Top 5: [('mañana', 2), ('clase', 2), ('procesamiento', 2), ('dato', 2), ('vos', 1)]

--- C (ajustado) ---
  Tokens totales:  28
  Tokens únicos:   24
  Top 5: [('mañana', 2), ('clase', 2), ('procesamiento', 2), ('dato', 2), ('vos', 1)]


---
## 4. Análisis: qué cambia entre pipelines

Tomate un minuto para reflexionar sobre las diferencias. Estas preguntas te ayudan a organizar la observación:

1. **¿Cambian los resultados entre pipelines?** Compará la cantidad de tokens totales y únicos. ¿Qué pipeline produce la lista más corta? ¿Por qué?

2. **¿Qué información se pierde en cada paso?**
   - El Pipeline A conserva stopwords ("de", "la", "que"), que dominan el ranking de frecuencia pero aportan poca información sobre el contenido.
   - El Pipeline B elimina stopwords, pero los lemas pueden ser incorrectos para formas del voseo.
   - El Pipeline C corrige esos lemas, pero depende de un diccionario construido a mano.

3. **¿Qué representación parece más útil?** Depende de la tarea. Para un conteo de frecuencias orientado al contenido, el Pipeline C da resultados más limpios. Pero si la tarea es estudiar estructura gramatical, eliminar stopwords puede ser contraproducente.

No existe un pipeline universalmente correcto. Cada decisión de preprocesamiento es una hipótesis sobre qué importa y qué no.

---
## 5. Propagación de errores: cómo un lema incorrecto altera los resultados

En el cuaderno `004` vimos que spaCy no lematiza bien las formas del voseo. Acá vamos a medir el impacto concreto de ese error sobre las frecuencias.

Para hacerlo más claro, vamos a usar un texto con mayor densidad de formas verbales del voseo, de modo que el efecto sea más visible.

In [21]:
# Texto con varias formas verbales del voseo
texto_voseo = (
    "Si vos querés, podés venir mañana. Pero si no podés, avisá. "
    "Yo sé que vos sabés de qué se trata. Vos siempre sabés. "
    "Además podés traer lo que quieras. Si querés, venís."
)

doc_voseo = nlp(texto_voseo)

# Pipeline B: lemas directos de spaCy (sin corrección)
lemas_spacy = [
    token.lemma_.lower()
    for token in doc_voseo
    if token.is_alpha and not token.is_stop
]

# Pipeline C: lemas corregidos
lemas_corregidos = pipeline_c(doc_voseo, CORRECCIONES_VOSEO)

print("--- Frecuencias con lemas de spaCy (Pipeline B) ---")
freq_spacy = Counter(lemas_spacy)
for palabra, cuenta in freq_spacy.most_common(10):
    print(f"  {palabra:20} -> {cuenta}")

print("\n--- Frecuencias con lemas corregidos (Pipeline C) ---")
freq_corregido = Counter(lemas_corregidos)
for palabra, cuenta in freq_corregido.most_common(10):
    print(f"  {palabra:20} -> {cuenta}")

--- Frecuencias con lemas de spaCy (Pipeline B) ---
  podés                -> 3
  vo                   -> 2
  querés               -> 2
  venir                -> 1
  mañana               -> 1
  avisá                -> 1
  sabé                 -> 1
  vos                  -> 1
  sabés                -> 1
  traer                -> 1

--- Frecuencias con lemas corregidos (Pipeline C) ---
  poder                -> 3
  vo                   -> 2
  querer               -> 2
  venir                -> 2
  saber                -> 2
  mañana               -> 1
  avisar               -> 1
  vos                  -> 1
  traer                -> 1
  quieras              -> 1


Fijate en la diferencia: sin corrección, `"sabés"` y `"sabé"` aparecen como dos lemas distintos (el modelo los trata como palabras diferentes). Con corrección, ambos se agrupan bajo `"saber"` con su frecuencia real. Lo mismo pasa con `"querer"` y `"poder"`.

Ahora veamos exactamente en qué tokens difieren los dos pipelines.

In [22]:
# Comparación token a token para ver dónde difieren
print("--- Detalle: dónde difieren los lemas ---")
print(f"{'TOKEN':<15} {'LEMA SPACY':<20} {'LEMA CORREGIDO':<20} {'¿DIFIERE?'}")
print("-" * 70)

for token in doc_voseo:
    if not token.is_alpha or token.is_stop:
        continue

    lema_spacy = token.lemma_.lower()
    forma = token.text.lower()
    lema_manual = CORRECCIONES_VOSEO.get(forma, lema_spacy)

    difiere = "<-- SÍ" if lema_spacy != lema_manual else ""
    print(f"  {token.text:<15} {lema_spacy:<20} {lema_manual:<20} {difiere}")

--- Detalle: dónde difieren los lemas ---
TOKEN           LEMA SPACY           LEMA CORREGIDO       ¿DIFIERE?
----------------------------------------------------------------------
  vos             vo                   vo                   
  querés          querés               querer               <-- SÍ
  podés           podés                poder                <-- SÍ
  venir           venir                venir                
  mañana          mañana               mañana               
  podés           podés                poder                <-- SÍ
  avisá           avisá                avisar               <-- SÍ
  vos             vo                   vo                   
  sabés           sabé                 saber                <-- SÍ
  Vos             vos                  vos                  
  sabés           sabés                saber                <-- SÍ
  podés           podés                poder                <-- SÍ
  traer           traer                traer 

Observá cómo el error de lematización se **propaga** al conteo de frecuencias:
- Sin corrección, formas como `"querés"`, `"podés"` y `"sabés"` aparecen con lemas incorrectos y se dispersan en el conteo.
- Con corrección, se agrupan correctamente bajo `"querer"`, `"poder"` y `"saber"`.

Este efecto se amplifica en corpus grandes. Un error de lema que parece menor en una oración se convierte en un sesgo sistemático cuando se repite miles de veces.

---
## 6. Comparación práctica: spaCy vs. Stanza

Hasta acá usamos exclusivamente spaCy. Pero no es la única herramienta que puede tokenizar y lematizar español. **Stanza**, desarrollada por el Stanford NLP Group, es otra librería que ofrece análisis morfológico con modelos entrenados sobre corpus distintos.

¿Por qué importa comparar? Porque si dos herramientas producen lemas diferentes para el mismo texto, la representación final cambia, y con ella los resultados de cualquier análisis posterior. Vamos a verlo en la práctica.

In [23]:
# Instalación de Stanza y spacy-stanza (ejecutar solo la primera vez)
!pip install stanza spacy-stanza -q

Cargamos el modelo de Stanza para español. El bloque incluye un *wrapper* de compatibilidad (`cargar_con_torch_compatible`) necesario porque PyTorch 2.6+ cambió el comportamiento por defecto de `torch.load`, lo que rompe la carga de los checkpoints de Stanza. Si corrés esto sin el wrapper, el proceso falla silenciosamente en algunas configuraciones.

In [24]:
import torch
import stanza

def cargar_con_torch_compatible(factory, *args, **kwargs):
    """Fuerza weights_only=False para checkpoints confiables de Stanza."""
    torch_load_original = torch.load

    def torch_load_compatible(*load_args, **load_kwargs):
        load_kwargs.setdefault("weights_only", False)
        return torch_load_original(*load_args, **load_kwargs)

    torch.load = torch_load_compatible
    try:
        return factory(*args, **kwargs)
    finally:
        torch.load = torch_load_original

# Descarga del modelo en español (solo la primera vez, puede tardar un minuto)
stanza.download("es", verbose=False)

# PyTorch 2.6+ cambió el valor por defecto de torch.load y rompe estos checkpoints
nlp_stanza = cargar_con_torch_compatible(
    stanza.Pipeline,
    "es",
    processors="tokenize,mwt,pos,lemma",
    verbose=False,
)
print("Modelo de Stanza cargado.")

Modelo de Stanza cargado.


### 6.1 Comparación de lemas: spaCy vs. Stanza

Procesamos el mismo texto con ambas herramientas y comparamos los lemas que asigna cada una. Prestá especial atención a las formas del voseo.

Vas a ver que Stanza tiene diferente comportamiento frente a formas como `"sabés"`, `"querés"` o `"podés"`.

In [26]:
# Texto de prueba con formas del voseo
texto_comparacion = "Vos sabés que si querés podés venir. Avisá si no venís."

# Procesamiento con spaCy
doc_spacy = nlp(texto_comparacion)

# Procesamiento con Stanza
doc_stanza = nlp_stanza(texto_comparacion)

# Resultados de spaCy
print("=" * 65)
print("spaCy")
print("=" * 65)
print(f"{'TOKEN':<15} {'LEMA':<15} {'POS':<10}")
print("-" * 40)
for token in doc_spacy:
    if not token.is_punct and not token.is_space:
        print(f"  {token.text:<15} {token.lemma_:<15} {token.pos_:<10}")

# Resultados de Stanza
print(f"\n{'=' * 65}")
print("Stanza")
print("=" * 65)
print(f"{'TOKEN':<15} {'LEMA':<15} {'POS':<10}")
print("-" * 40)
for oracion in doc_stanza.sentences:
    for token in oracion.words:
        if token.upos != "PUNCT":
            print(f"  {token.text:<15} {token.lemma:<15} {token.upos:<10}")

spaCy
TOKEN           LEMA            POS       
----------------------------------------
  Vos             vos             ADV       
  sabés           sabé            NOUN      
  que             que             SCONJ     
  si              si              SCONJ     
  querés          querés          NOUN      
  podés           podés           ADJ       
  venir           venir           VERB      
  Avisá           Avisá           PROPN     
  si              si              SCONJ     
  no              no              ADV       
  venís           venís           VERB      

Stanza
TOKEN           LEMA            POS       
----------------------------------------
  Vos             yo              PRON      
  sabés           saber           VERB      
  que             que             SCONJ     
  si              si              SCONJ     
  querés          querer          VERB      
  podés           poder           AUX       
  venir           venir           VERB      
  Avisá 

Las diferencias son notables. Stanza lematiza correctamente `"sabés"` → `"saber"`, `"querés"` → `"querer"`, `"podés"` → `"poder"`, `"venís"` → `"venir"`. spaCy, en cambio, devuelve formas incorrectas o las trata como sustantivos (`NOUN`) o adjetivos (`ADJ`).

Sin embargo, Stanza también tiene sus limitaciones: lematiza `"Vos"` como `"yo"`, que es incorrecto desde el punto de vista del español rioplatense.

### 6.2 Comparación lado a lado

Para ver las diferencias con más claridad, armemos una tabla que ponga los lemas de ambas herramientas uno al lado del otro.

In [27]:
# Extraemos lemas de spaCy (sin puntuación)
lemas_sp = [
    (token.text, token.lemma_, token.pos_)
    for token in doc_spacy
    if not token.is_punct and not token.is_space
]

# Extraemos lemas de Stanza (sin puntuación)
lemas_st = [
    (word.text, word.lemma, word.upos)
    for sent in doc_stanza.sentences
    for word in sent.words
    if word.upos != "PUNCT"
]

print(f"{'TOKEN':<12} {'LEMA spaCy':<15} {'LEMA Stanza':<15} {'¿DIFIERE?'}")
print("-" * 55)

# Comparamos posición a posición
for i in range(min(len(lemas_sp), len(lemas_st))):
    texto_sp, lema_sp, _ = lemas_sp[i]
    texto_st, lema_st, _ = lemas_st[i]

    difiere = "<-- SÍ" if lema_sp.lower() != lema_st.lower() else ""
    print(f"  {texto_sp:<12} {lema_sp:<15} {lema_st:<15} {difiere}")

TOKEN        LEMA spaCy      LEMA Stanza     ¿DIFIERE?
-------------------------------------------------------
  Vos          vos             yo              <-- SÍ
  sabés        sabé            saber           <-- SÍ
  que          que             que             
  si           si              si              
  querés       querés          querer          <-- SÍ
  podés        podés           poder           <-- SÍ
  venir        venir           venir           
  Avisá        Avisá           avisar          <-- SÍ
  si           si              si              
  no           no              no              
  venís        venís           venir           <-- SÍ


De 11 tokens no puntados, 6 tienen lemas distintos entre las dos herramientas. Eso es más del 50% de divergencia en un texto corto con formas del voseo. En un corpus periodístico argentino, esa diferencia se acumula de manera significativa.

### 6.3 Impacto en un pipeline de preprocesamiento

Veamos cómo esas diferencias de lematización se traducen en listas de tokens distintas cuando aplicamos un pipeline tipo B (lemas + sin stopwords).

Nota: Stanza no incluye lista de stopwords propia, así que reutilizamos la de spaCy para que la comparación sea justa.

In [28]:
# Pipeline B con spaCy
tokens_spacy = [
    token.lemma_.lower()
    for token in doc_spacy
    if token.is_alpha and not token.is_stop
]

# Pipeline equivalente con Stanza
# Stanza no tiene is_stop incorporado, así que usamos la lista de spaCy
stopwords_es = nlp.Defaults.stop_words

tokens_stanza = [
    word.lemma.lower()
    for sent in doc_stanza.sentences
    for word in sent.words
    if word.upos != "PUNCT"
    and word.text.isalpha()
    and word.text.lower() not in stopwords_es
]

print("Pipeline B con spaCy:")
print(f"  Tokens: {tokens_spacy}")
print(f"  Únicos: {len(set(tokens_spacy))}")

print("\nPipeline B con Stanza:")
print(f"  Tokens: {tokens_stanza}")
print(f"  Únicos: {len(set(tokens_stanza))}")

# Diferencias
set_spacy = set(tokens_spacy)
set_stanza = set(tokens_stanza)

solo_spacy = set_spacy - set_stanza
solo_stanza = set_stanza - set_spacy

if solo_spacy or solo_stanza:
    print("\nDiferencias:")
    if solo_spacy:
        print(f"  Solo en spaCy:  {solo_spacy}")
    if solo_stanza:
        print(f"  Solo en Stanza: {solo_stanza}")
else:
    print("\nAmbas herramientas produjeron los mismos tokens únicos.")

Pipeline B con spaCy:
  Tokens: ['vos', 'sabé', 'querés', 'podés', 'venir', 'avisá', 'venís']
  Únicos: 7

Pipeline B con Stanza:
  Tokens: ['yo', 'saber', 'querer', 'poder', 'venir', 'avisar', 'venir']
  Únicos: 6

Diferencias:
  Solo en spaCy:  {'avisá', 'venís', 'querés', 'sabé', 'vos', 'podés'}
  Solo en Stanza: {'saber', 'querer', 'avisar', 'poder', 'yo'}


Los dos pipelines producen vocabularios completamente disjuntos para los verbos del voseo. Si usáramos esas listas para entrenar un clasificador o calcular similitudes entre documentos, los resultados serían incompatibles entre sí.

### 6.4 Integración: usar Stanza desde la API de spaCy

Si después de comparar decidís que Stanza lematiza mejor para tu caso, no hace falta reescribir todo tu código. La librería `spacy-stanza` permite usar modelos de Stanza directamente desde la interfaz de spaCy. Así conservás todo tu pipeline (filtros, `Matcher`, `EntityRuler`) pero con un motor de análisis distinto por debajo.

**Nota técnica:** al cargar el pipeline híbrido, es importante especificar solo los procesadores que necesitamos (`tokenize,mwt,pos,lemma`). Si se omite este parámetro, `spacy-stanza` intenta cargar todos los procesadores de Stanza (incluido el de *constituency parsing*), lo que puede generar errores de compatibilidad entre versiones.

In [29]:
import spacy_stanza

# Reutilizamos el wrapper de compatibilidad para la integración spaCy + Stanza
nlp_hibrido = cargar_con_torch_compatible(
    spacy_stanza.load_pipeline,
    "es",
    processors="tokenize,mwt,pos,lemma",
    verbose=False,
)
print(f"Pipeline híbrido cargado: {nlp_hibrido.pipe_names}")

Pipeline híbrido cargado: []


El pipeline híbrido está listo. Ahora procesamos el mismo texto y verificamos que la API de spaCy se comporta exactamente igual, pero con los lemas de Stanza por debajo.

In [30]:
# Procesamos el mismo texto con el pipeline híbrido
doc_hibrido = nlp_hibrido(texto_comparacion)

print("--- Pipeline híbrido (Stanza vía spaCy) ---")
print(f"{'TOKEN':<15} {'LEMA':<15} {'POS':<10} {'is_stop'}")
print("-" * 50)
for token in doc_hibrido:
    if not token.is_punct and not token.is_space:
        print(f"  {token.text:<15} {token.lemma_:<15} {token.pos_:<10} {token.is_stop}")

--- Pipeline híbrido (Stanza vía spaCy) ---
TOKEN           LEMA            POS        is_stop
--------------------------------------------------
  Vos             yo              PRON       False
  sabés           saber           VERB       False
  que             que             SCONJ      True
  si              si              SCONJ      True
  querés          querer          VERB       False
  podés           poder           AUX        False
  venir           venir           VERB       False
  Avisá           avisar          VERB       False
  si              si              SCONJ      True
  no              no              ADV        True
  venís           venir           VERB       False


Observá que el pipeline híbrido se usa **exactamente igual** que spaCy puro: `token.text`, `token.lemma_`, `token.pos_`, `token.is_stop`. Esto significa que podés intercambiar el motor de análisis sin tocar el resto de tu código.

Esta es una ventaja arquitectónica importante: **separar la interfaz del motor** permite experimentar con distintas herramientas sin reescribir pipelines completos.

### 6.5 Resumen de la comparación

| Aspecto | spaCy (`es_core_news_sm`) | Stanza (`es`) |
|---|---|---|
| **Corpus de entrenamiento** | AnCora (predominantemente peninsular) | AnCora + UD Spanish (varias fuentes) |
| **Voseo** | Lematización frecuentemente incorrecta | Puede manejar mejor algunas formas |
| **Stopwords** | Lista incorporada (`token.is_stop`) | No incluye lista propia |
| **Velocidad** | Más rápido en producción | Más lento (modelos neuronales más pesados) |
| **Integración** | API nativa muy completa | Integrable vía `spacy-stanza` |

La conclusión no es que una herramienta sea mejor que la otra. Es que **la elección de herramienta es otra decisión de preprocesamiento**, y cambia los resultados. Documentar qué herramienta y qué versión usaste es parte de la reproducibilidad del trabajo.

---
## 7. Formalización: un texto procesado es una lista de tokens

Después de todo el preprocesamiento, lo que queda es una estructura simple:

> **Un texto procesado = una lista ordenada de tokens (strings)**

Esta es la representación que vamos a usar como punto de partida en los próximos cuadernos, cuando necesitemos contar frecuencias, comparar documentos o construir matrices de términos.

La lista no contiene información sobre el orden original de las oraciones ni sobre relaciones sintácticas. Esa pérdida es deliberada: es el precio que pagamos por obtener una representación que se pueda operar numéricamente.

In [31]:
# Formalización: el texto como lista de tokens
texto_procesado = pipeline_c(doc, CORRECCIONES_VOSEO)

print("REPRESENTACIÓN FINAL DEL TEXTO:")
print(f"  Tipo: {type(texto_procesado)}")
print(f"  Largo: {len(texto_procesado)} tokens")
print(f"  Contenido: {texto_procesado}")
print()
print("Esta lista es la entrada para cualquier análisis cuantitativo posterior.")

REPRESENTACIÓN FINAL DEL TEXTO:
  Tipo: <class 'list'>
  Largo: 28 tokens
  Contenido: ['vos', 'saber', 'querer', 'venir', 'mañana', 'clase', 'nlp', 'clase', 'procesamiento', 'texto', 'representación', 'dato', 'tener', 'traer', 'notebook', 'spacy', 'instalado', 'poder', 'mañana', 'avisar', 'coordinar', 'procesamiento', 'lenguaje', 'natural', 'clave', 'entender', 'dato', 'textual']

Esta lista es la entrada para cualquier análisis cuantitativo posterior.


---
## 8. Ejercicio práctico

### Consigna

Tomá el siguiente texto (un fragmento adaptado de una noticia argentina) y realizá las actividades indicadas.

**Texto:**

> El Gobierno nacional anunció un nuevo paquete de medidas económicas para el sector agroindustrial. La iniciativa busca promover la exportación de productos con valor agregado y reducir la presión impositiva sobre las economías regionales. Según explicaron desde el Ministerio de Economía, las medidas entran en vigencia la semana que viene. Los representantes del campo manifestaron su apoyo pero pidieron que se incluyan también beneficios para los pequeños productores.

### Actividades

1. Procesá el texto con spaCy.
2. Construí **dos pipelines distintos** (elegí entre los que vimos o inventá el tuyo).
3. Para cada pipeline, mostrá la lista de tokens resultante y el top 10 de frecuencias.
4. **Justificá** en una celda Markdown cuál de los dos usarías para un análisis de contenido y por qué.

### Para pensar antes de arrancar

- ¿Qué información del texto esperás que sea relevante para un análisis de contenido?
- ¿Hay formas verbales en este texto que el modelo podría lematizar mal?
- ¿Qué pasaría si aplicaras el Pipeline A sobre este texto? ¿Qué palabras dominarían el ranking?

In [32]:
# Texto para el ejercicio
texto_ejercicio = (
    "El Gobierno nacional anunció un nuevo paquete de medidas económicas "
    "para el sector agroindustrial. La iniciativa busca promover la "
    "exportación de productos con valor agregado y reducir la presión "
    "impositiva sobre las economías regionales. Según explicaron desde "
    "el Ministerio de Economía, las medidas entran en vigencia la semana "
    "que viene. Los representantes del campo manifestaron su apoyo pero "
    "pidieron que se incluyan también beneficios para los pequeños productores."
)

# Tu código acá
doc_ejercicio = nlp(texto_ejercicio)
pipeline_b_resultado = pipeline_b(doc_ejercicio)
print("Resultado del Pipeline B para el ejercicio:")
print(pipeline_b_resultado) 
pipeline_c_resultado = pipeline_c(doc_ejercicio, CORRECCIONES_VOSEO)
print("\nResultado del Pipeline C para el ejercicio:")
print(pipeline_c_resultado) 

freq_b_ejercicio = Counter(pipeline_b_resultado)
print("\nTop 10 palabras más frecuentes en el ejercicio (Pipeline B):")
for palabra, cuenta in freq_b_ejercicio.most_common(10):
    print(f"  {palabra:20} -> {cuenta}")

freq_c_ejercicio = Counter(pipeline_c_resultado)
print("\nTop 10 palabras más frecuentes en el ejercicio (Pipeline C):")
for palabra, cuenta in freq_c_ejercicio.most_common(10):
    print(f"  {palabra:20} -> {cuenta}")
# ...

Resultado del Pipeline B para el ejercicio:
['gobierno', 'nacional', 'anunciar', 'paquete', 'medida', 'económico', 'sector', 'agroindustrial', 'iniciativa', 'buscar', 'promover', 'exportación', 'producto', 'valor', 'agregado', 'reducir', 'presión', 'impositivo', 'economía', 'regional', 'explicar', 'ministerio', 'economía', 'medida', 'entrar', 'vigencia', 'semana', 'venir', 'representante', 'campo', 'manifestar', 'apoyo', 'pedir', 'incluir', 'beneficio', 'pequeño', 'productor']

Resultado del Pipeline C para el ejercicio:
['gobierno', 'nacional', 'anunciar', 'paquete', 'medida', 'económico', 'sector', 'agroindustrial', 'iniciativa', 'buscar', 'promover', 'exportación', 'producto', 'valor', 'agregado', 'reducir', 'presión', 'impositivo', 'economía', 'regional', 'explicar', 'ministerio', 'economía', 'medida', 'entrar', 'vigencia', 'semana', 'venir', 'representante', 'campo', 'manifestar', 'apoyo', 'pedir', 'incluir', 'beneficio', 'pequeño', 'productor']

Top 10 palabras más frecuentes en 

*Escribí tu justificación acá.*

Para este texto ambos pipelines producen el mismo resultado porque no contiene formas verbales del voseo. El Pipeline C agrega valor cuando el corpus tiene texto rioplatense coloquial. Para un análisis de contenido periodístico formal como este, el Pipeline B es suficiente — lematiza correctamente y elimina stopwords sin necesitar correcciones manuales. Las palabras más relevantes ("medida", "economía", "gobierno") quedan bien representadas en ambos casos.

---
## Cierre

En este cuaderno formalizamos algo que puede parecer trivial pero tiene consecuencias profundas: **un texto procesado es una lista de tokens, y esa lista depende de las decisiones que tomamos al construirla**.

Ideas centrales para llevarte:
- No existe un pipeline de preprocesamiento universalmente correcto.
- Cada decisión (lowercase, lematización, filtrado de stopwords) descarta información.
- Los errores del modelo (como la lematización incorrecta del voseo) se propagan silenciosamente al análisis.
- Herramientas distintas (spaCy, Stanza) producen representaciones distintas del mismo texto, y se pueden combinar vía `spacy-stanza`.
- Documentar el pipeline y la herramienta usados es tan importante como documentar los resultados.

En el próximo cuaderno vamos a usar estas listas de tokens como insumo para construir representaciones numéricas de los textos (TF-IDF).